# XLSR Pipeline — French ASR + Direct Speech Translation (FR → EN)
### Applied to the CFPP2000 Corpus (Anita Musso, 11th arrondissement)

| Stage | Model | Task |
|-------|-------|------|
| A | `jonatasgrosman/wav2vec2-large-xlsr-53-french` | ASR — French transcription |
| B | `facebook/wav2vec2-xls-r-1b-21-to-en` | ST — direct audio → English translation |

**Hardware:** Stage B requires a GPU with ≥14 GB VRAM. A free Colab T4 runtime is sufficient.  
The two models are loaded and released sequentially to stay within VRAM limits.

**Input:** `.wav` files archived on Zenodo: https://doi.org/10.5281/zenodo.19479351  
**Output:** `output/transcriptions_xlsr.csv` with columns `fichier`, `transcription_fr`, `translation_en`

**Data source:** [CFPP2000](http://cfpp2000.univ-paris3.fr) — publicly available under CC BY-NC-SA 4.0.  
See `data/README.md` for full provenance.

---

---
## Cell 1 — Environment check

In [ ]:
# ── 1. Environment check ───────────────────────────────────────────────────────
import torch
import psutil

if torch.cuda.is_available():
    name  = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU  : {name}  —  {total:.1f} GB VRAM")
else:
    print("⚠  No GPU detected — inference will be very slow on CPU.")
    print("   For Stage B (XLS-R 1B), a GPU with ≥14 GB VRAM is strongly recommended.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

ram = psutil.virtual_memory()
print(f"RAM  : {ram.total/1024**3:.1f} GB total  /  {ram.available/1024**3:.1f} GB available")

---
## Cell 2 — Install dependencies
Skip if your environment already has the packages in `requirements.txt`.

In [ ]:
# ── 2. Dependencies ────────────────────────────────────────────────────────────
# Pinned to transformers==4.44.2 for XLS-R ST model compatibility.
%pip install -q transformers==4.44.2 sentencepiece librosa soundfile tqdm accelerate requests psutil

---
## Cell 3 — Configuration

In [ ]:
# ── 3. Configuration ───────────────────────────────────────────────────────────
from pathlib import Path

AUDIO_DIR = Path("data/segments")   # populated by cell 0
OUT_DIR   = Path("output")
SR        = 16_000  # required sample rate (Hz) for both models
CHUNK_S   = 30      # max chunk duration fed to each model (seconds)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Audio dir exists : {AUDIO_DIR.exists()}")
print(f"Output dir       : {OUT_DIR.resolve()}")

---
## Cell 4 — Imports and audio utilities

In [ ]:
# ── 4. Imports and utilities ───────────────────────────────────────────────────
import pandas as pd
import gc
import numpy as np
import librosa
from tqdm import tqdm
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor


def load_16k_mono(path, sr: int = SR) -> np.ndarray:
    """Load an audio file, resample to sr Hz, convert to mono."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y


def chunk_audio(y: np.ndarray, sr: int = SR,
                chunk_s: float = CHUNK_S, overlap_s: float = 1.0):
    """Split a signal into overlapping chunks for long-form inference."""
    hop = int((chunk_s - overlap_s) * sr)
    win = int(chunk_s * sr)
    if hop <= 0 or len(y) <= win:
        return [y]
    chunks, i = [], 0
    while i < len(y):
        seg = y[i : i + win]
        if len(seg) == 0:
            break
        chunks.append(seg)
        i += hop
    return chunks


def clean(s: str) -> str:
    """Collapse redundant whitespace."""
    return " ".join(s.strip().split())


def free_gpu():
    """Release VRAM between model stages."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


wavs = sorted(AUDIO_DIR.glob("*.wav"))
print(f"{len(wavs)} .wav file(s) found in {AUDIO_DIR}")

---
## Stage A — French ASR (XLSR-53)

In [ ]:
# ── 5. Load ASR model ──────────────────────────────────────────────────────────
ASR_ID        = "jonatasgrosman/wav2vec2-large-xlsr-53-french"
asr_processor = Wav2Vec2Processor.from_pretrained(ASR_ID)
asr_model     = Wav2Vec2ForCTC.from_pretrained(ASR_ID).to(DEVICE).eval()
print(f"ASR model loaded on {DEVICE}")

In [ ]:
# ── 6. Run ASR transcription ───────────────────────────────────────────────────
@torch.inference_mode()
def asr_fr(y: np.ndarray) -> str:
    texts = []
    for chunk in chunk_audio(y):
        inp          = asr_processor(chunk, sampling_rate=SR, return_tensors="pt")
        input_values = inp.input_values.to(DEVICE)
        attn         = inp.get("attention_mask")
        if attn is not None:
            attn = attn.to(DEVICE)
        logits = asr_model(input_values, attention_mask=attn).logits
        ids    = torch.argmax(logits, dim=-1)
        texts.append(asr_processor.decode(ids[0], skip_special_tokens=True))
    return clean(" ".join(texts))


asr_results = {}  # {stem: transcription_fr}

for wav in tqdm(wavs, desc="ASR FR"):
    try:
        y = load_16k_mono(wav)
        asr_results[wav.stem] = asr_fr(y)
    except Exception as e:
        print(f"  ✗ {wav.name}: {e}")
        asr_results[wav.stem] = ""

print(f"\n{len(asr_results)} transcription(s) complete.")

In [ ]:
# ── 7. Free VRAM before Stage B ────────────────────────────────────────────────
del asr_model, asr_processor
free_gpu()
print("VRAM freed.")

---
## Stage B — Direct Speech Translation FR → EN (XLS-R 1B)

In [ ]:
# ── 8. Load ST model ───────────────────────────────────────────────────────────
from transformers import Wav2Vec2FeatureExtractor, MBart50TokenizerFast, SpeechEncoderDecoderModel

ST_ID             = "facebook/wav2vec2-xls-r-1b-21-to-en"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(ST_ID)
tokenizer         = MBart50TokenizerFast.from_pretrained(ST_ID)
st_model          = SpeechEncoderDecoderModel.from_pretrained(ST_ID).to(DEVICE).eval()
print(f"ST model loaded on {DEVICE}")

In [ ]:
# ── 9. Run direct speech translation ──────────────────────────────────────────
@torch.inference_mode()
def translate_fr_en(y: np.ndarray) -> str:
    texts = []
    for chunk in chunk_audio(y):
        inp          = feature_extractor(chunk, sampling_rate=SR, return_tensors="pt")
        input_values = inp.input_values.to(DEVICE)
        attn         = inp.get("attention_mask")
        if attn is not None:
            attn = attn.to(DEVICE)
        generated = st_model.generate(
            input_values,
            attention_mask=attn,
            num_beams=4,
            max_length=512,
            no_repeat_ngram_size=3,   # prevents trigram repetition
            repetition_penalty=1.3,   # penalises already-generated tokens
            early_stopping=True,
        )
        txt = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
        texts.append(txt.strip())
    return clean(" ".join(texts))


st_results = {}  # {stem: translation_en}

for wav in tqdm(wavs, desc="ST FR→EN"):
    try:
        y = load_16k_mono(wav)
        st_results[wav.stem] = translate_fr_en(y)
    except Exception as e:
        print(f"  ✗ {wav.name}: {e}")
        st_results[wav.stem] = ""

print(f"\n{len(st_results)} translation(s) complete.")

In [ ]:
# ── 10. Free VRAM ──────────────────────────────────────────────────────────────
del st_model, feature_extractor, tokenizer
free_gpu()
print("VRAM freed.")

---
## Stage C — Preview and export

In [ ]:
# ── 11. Preview first 5 results ────────────────────────────────────────────────
for wav in wavs[:5]:
    print(f"📁 {wav.name}")
    print(f"   FR : {asr_results.get(wav.stem, '(empty)')}")
    print(f"   EN : {st_results.get(wav.stem, '(empty)')}")
    print()

In [ ]:
# ── 12. Export CSV ─────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for wav in wavs:
    rows.append({
        "fichier"         : wav.name,
        "transcription_fr": asr_results.get(wav.stem, ""),
        "translation_en"  : st_results.get(wav.stem, ""),
    })

df      = pd.DataFrame(rows)
out_csv = OUT_DIR / "transcriptions_xlsr.csv"
df.to_csv(out_csv, index=False, encoding="utf-8")

print(f"CSV saved → {out_csv}  ({len(df)} rows)")
df.head()